In [14]:
# --- BLOQUE 0: INFRAESTRUCTURA ---
import sys
import subprocess

# Instalación robusta de dependencias
# Utilizamos !pip install directamente para asegurar que se instalen todos los paquetes necesarios.
!{sys.executable} -m pip install astropy scipy matplotlib pandas requests

import numpy as np
import matplotlib.pyplot as plt
import requests
import io
from astropy.io import fits
from astropy.cosmology import Planck18 as cosmo
from astropy import units as u
from scipy.optimize import curve_fit

print("[SYSTEM] Iniciando Protocolo ASTRO-MINER v2 (Fermi-LAT)...")

# --- BLOQUE 1: EXTRACCIÓN SOBERANA DE DATOS (NASA FSSC) ---
# URL ACTUALIZADA Y VERIFICADA (3rd GRB Catalog - intentar nombre genérico)
# Las versiones específicas (v02, v03, v04, v05) han resultado inestables.
# Se intenta usar 'gll_grb_cat.fit' que a menudo apunta a la última versión disponible.
# Si esto falla, la URL debe ser verificada manualmente en el sitio del FSSC.
CATALOG_URL = "https://fermi.gsfc.nasa.gov/ssc/data/access/lat/msl_grb_catalog/gll_grb_cat.fit"

print(f"[NET] Descargando catálogo oficial desde: {CATALOG_URL}")

try:
    response = requests.get(CATALOG_URL)
    response.raise_for_status() # Lanza error si es 404

    # Decodificación en memoria RAM
    hdul = fits.open(io.BytesIO(response.content))
    print("[SYSTEM] Archivo FITS decodificado en memoria.")

    # La tabla de datos suele estar en la extensión 1
    data = hdul[1].data
    cols = hdul[1].columns

    # DEBUG: Imprimir columnas si hay duda
    # print("[DEBUG] Columnas disponibles:", cols.names)

except Exception as e:
    print(f"[ERROR] Fallo crítico en descarga o lectura: {e}")
    print("[HINT] Si es 404, la NASA movió el archivo. Verificar: https://fermi.gsfc.nasa.gov/ssc/data/access/lat/msl_grb_catalog/")
    sys.exit()

# --- BLOQUE 2: MINERÍA Y FILTRADO ---
print("[MINING] Filtrando eventos con redshift espectroscópico...")

# Mapeo de columnas (Asumimos consistencia en los nombres de columnas)
try:
    names = data['GRB_NAM'] # Nombre del GRB
    redshifts = data['REDSHIFT'] # Redshift espectroscópico

    # Tiempos y Energías para el fotón de más alta energía
    t_high_E = data['HIGHEST_ENERGY_TIME'] # Tiempo de llegada del fotón más energético (s)
    E_max_MeV = data['HIGHEST_ENERGY'] # Energía del fotón más energético (MeV)

except KeyError as e:
    print(f"[ERROR] Columna no encontrada: {e}")
    print("[DEBUG] Las columnas disponibles son:", cols.names)
    sys.exit()

E_max_GeV = E_max_MeV / 1000.0

# Arrays para datos limpios
clean_z = []
clean_dt = []
clean_E = []
clean_names = []

# Filtrado estricto
count_rejected = 0
for i in range(len(names)):
    z = redshifts[i]
    t = t_high_E[i]
    e = E_max_GeV[i]

    # Criterios físicos de calidad:
    # 1. Redshift válido (z > 0 y finito)
    # 2. Energía significativa (> 3 GeV para tener palanca real en la búsqueda de LIV)
    # 3. Tiempo post-trigger (t > 0)
    if z > 0 and np.isfinite(z) and e > 3.0: # Subimos el umbral a 3 GeV para limpiar ruido
        clean_z.append(z)
        clean_dt.append(t)
        clean_E.append(e)
        clean_names.append(names[i])
    else:
        count_rejected += 1

clean_z = np.array(clean_z)
clean_dt = np.array(clean_dt)
clean_E = np.array(clean_E)

print(f"[DATA] Total eventos en catálogo: {len(names)}")
print(f"[DATA] Eventos filtrados para física (z>0, E>3GeV): {len(clean_z)}")

if len(clean_z) == 0:
    print("[FAIL] No hay eventos que cumplan los criterios. Baja el umbral de energía.")
    sys.exit()

# --- BLOQUE 3: CÁLCULO COSMOLÓGICO (FACTOR K) ---
print("[PHYS] Calculando distancias cosmológicas relativistas (K-factor)....")

def calculate_K_factor_approx(z_array):
    # Aproximación de la integral cosmológica para LIV
    # K(z) ~ \int (1+z)/H(z) dz
    # Usamos astropy para la distancia de luminosidad y convertimos
    # O hacemos la integral numérica rápida

    Ks = []
    # Pre-calculamos H0 para normalizar
    H0 = cosmo.H0.value

    for z in z_array:
        # Integral numérica simple
        z_grid = np.linspace(0, z, 50)
        # E(z) = H(z)/H0 = sqrt(Om(1+z)^3 + Ol)
        Ez = np.sqrt(cosmo.Om0 * (1+z_grid)**3 + cosmo.Ode0)
        integrand = (1 + z_grid) / Ez
        integral = np.trapz(integrand, z_grid)

        # El factor K es proporcional a esta integral
        Ks.append(integral)

    return np.array(Ks)

K_factors = calculate_K_factor_approx(clean_z)

# Normalización: tau_n = dt / E
lags_normalized = clean_dt / clean_E # s / GeV

# --- BLOQUE 4: REGRESIÓN (Búsqueda de m_QG) ---
def linear_model(x, m, c):
    return m * x + c

# Ajuste robusto (ignorando outliers extremos si es necesario, aquí simple)
popt, pcov = curve_fit(linear_model, K_factors, lags_normalized)
m_fit = popt[0]
c_fit = popt[1]
m_err = np.sqrt(np.diag(pcov))[0]

print(f"[RESULT] Pendiente LIV (m): {m_fit:.4e} +/- {m_err:.4e} s/GeV")

# Interpretación rápida
if m_fit > 2 * m_err:
    print(">>> POSIBLE SEÑAL DETECTADA: Pendiente positiva significativa.")
elif m_fit < -2 * m_err:
    print(">>> ANOMALÍA NEGATIVA: Los fotones de alta energía llegan ANTES (?!).")
else:
    print(">>> RESULTADO NULO: Compatible con Relatividad General (m=0).")

# --- BLOQUE 5: VISUALIZACIÓN ---
plt.figure(figsize=(12, 7))
plt.style.use('dark_background')

# Scatter
plt.errorbar(K_factors, lags_normalized, yerr=0.1*lags_normalized, fmt='o',
             color='#00ffcc', ecolor='gray', alpha=0.8, label='Datos Fermi-LAT')

# Ajuste
x_range = np.linspace(0, max(K_factors)*1.1, 100)
plt.plot(x_range, linear_model(x_range, m_fit, c_fit),
         color='#ff00ff', linestyle='--', linewidth=2, label=f'Ajuste (m={m_fit:.2e})')

# Einstein
plt.axhline(y=0, color='white', linestyle=':', label='Einstein (m=0)')

plt.xlabel(r'Factor de Distancia Cosmológica $K(z)$')
plt.ylabel(r'Retraso Normalizado $\Delta t / E$ (s/GeV)')
plt.title('TEST DE TIEMPO DE VUELO: Buscando Fractura en Lorentz\nDatos: Fermi-LAT 3rd GRB Catalog', color='white')
plt.legend()
plt.grid(True, alpha=0.2)

# Etiquetas inteligentes
sorted_indices = np.argsort(clean_E)[::-1] # Índices de mayor a menor energía
for i in sorted_indices[:5]: # Etiquetar los top 5 más energéticos
    plt.annotate(f"{clean_names[i]}\n({clean_E[i]:.1f} GeV)",
                 (K_factors[i], lags_normalized[i]),
                 color='yellow', fontsize=9, xytext=(5, 5), textcoords='offset points')

plt.tight_layout()
plt.show()

[SYSTEM] Iniciando Protocolo ASTRO-MINER v2 (Fermi-LAT)...
[NET] Descargando catálogo oficial desde: https://fermi.gsfc.nasa.gov/ssc/data/access/lat/msl_grb_catalog/gll_grb_cat.fit


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



[ERROR] Fallo crítico en descarga o lectura: 404 Client Error: Not Found for url: https://fermi.gsfc.nasa.gov/ssc/data/access/lat/msl_grb_catalog/gll_grb_cat.fit
[HINT] Si es 404, la NASA movió el archivo. Verificar: https://fermi.gsfc.nasa.gov/ssc/data/access/lat/msl_grb_catalog/
Traceback (most recent call last):
  File "/tmp/ipython-input-4018987778.py", line 31, in <cell line: 0>
    response.raise_for_status() # Lanza error si es 404
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 404 Client Error: Not Found for url: https://fermi.gsfc.nasa.gov/ssc/data/access/lat/msl_grb_catalog/gll_grb_cat.fit

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_o

TypeError: object of type 'NoneType' has no len()

In [15]:
# --- BLOQUE 0: INFRAESTRUCTURA Y DEPENDENCIAS ---
import sys
import subprocess
import warnings

# Silenciar advertencias no críticas de FITS
warnings.filterwarnings('ignore')

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

try:
    import astropy
    import scipy
    import pandas
    import requests
except ImportError:
    print("[SYSTEM] Instalando librerías astronómicas críticas...")
    install("astropy")
    install("scipy")
    install("matplotlib")
    install("pandas")
    install("requests")

import numpy as np
import matplotlib.pyplot as plt
import requests
import io
from astropy.io import fits
from astropy.cosmology import Planck18 as cosmo
from astropy import units as u # Added for units.s**-1
from scipy.optimize import curve_fit

print("[SYSTEM] Iniciando Protocolo ASTRO-MINER v3 (Multi-Target)...")

# --- BLOQUE 1: EXTRACCIÓN SOBERANA DE DATOS (ESTRATEGIA REDUNDANTE) ---
# Lista de URLs candidatas ordenadas por preferencia (versiones más nuevas primero)
# Se han actualizado las URLs para reflejar los enlaces actuales del 3rd GRB Catalog (msl_grb_catalog).
# El nombre genérico 'gll_grb_cat.fit' a menudo apunta a la versión más reciente.
CANDIDATE_URLS = [
    "https://fermi.gsfc.nasa.gov/ssc/data/access/lat/msl_grb_catalog/gll_grb_cat.fit", # Genérico para 3FLGC
    "https://fermi.gsfc.nasa.gov/ssc/data/access/lat/msl_grb_catalog/gll_grb_cat_v05.fit" # Versión específica v05
]

hdul = None
data = None
used_url = ""

print(f"[NET] Iniciando barrido de servidores FSSC ({len(CANDIDATE_URLS)} objetivos)...")

for url in CANDIDATE_URLS:
    try:
        print(f"   -> Probando: {url.split('/')[-1]} ...", end=" ")
        response = requests.get(url, timeout=15) # Timeout para no colgarse
        if response.status_code == 200:
            hdul = fits.open(io.BytesIO(response.content))
            data = hdul[1].data
            used_url = url
            print("¡ÉXITO! [200 OK]")
            break
        else:
            print(f"FALLO [{response.status_code}]")
    except Exception as e:
        print(f"ERROR DE RED ({str(e)})")

if data is None:
    print("\n[CRITICAL ERROR] Ninguna URL del catálogo respondió. La NASA puede estar offline o haber reestructurado todo el directorio.")
    print("[ACTION] Verifica manualmente: https://fermi.gsfc.nasa.gov/ssc/data/access/lat/msl_grb_catalog/") # Updated hint
    sys.exit()

print(f"[SYSTEM] Datos cargados en memoria desde: {used_url}")

# --- BLOQUE 2: AUTO-MAPEO DE COLUMNAS (INTELIGENCIA DE DATOS) ---
# Detectamos los nombres de las columnas dinámicamente para evitar KeyErrors
cols = [c.name.upper() for c in hdul[1].columns]
print(f"[INFO] Columnas detectadas: {len(cols)}")

def find_col(candidates, column_list):
    for cand in candidates:
        if cand in column_list: return cand
    return None

# Diccionario de búsqueda heurística
col_map = {
    'NAME': find_col(['GRB_NAM', 'NAME', 'GRB_NAME'], cols),
    'Z': find_col(['REDSHIFT', 'Z'], cols),
    'TIME': find_col(['HIGHEST_ENERGY_TIME', 'GBM_T90_START'], cols), # Preferencia por fotón HE
    'ENERGY': find_col(['HIGHEST_ENERGY', 'ENERGY'], cols)
}

# Verificación de integridad
if not all(col_map.values()):
    print("[ERROR] No se pudieron identificar las columnas críticas automáticamente.")
    print(f"Mapa detectado: {col_map}")
    print(f"Columnas disponibles: {cols}")
    sys.exit()

print(f"[INFO] Mapeo exitoso: Z='{col_map['Z']}', E='{col_map['ENERGY']}'")

# --- BLOQUE 3: MINERÍA Y FÍSICA ---
# Extracción vectorial (más rápida que loops)
raw_z = data[col_map['Z']]
raw_t = data[col_map['TIME']]
raw_e = data[col_map['ENERGY']] # Usualmente MeV
raw_names = data[col_map['NAME']]

# Conversión de Energía a GeV
# Detectar unidades: si el max es > 10000, asumimos MeV. Si es < 1000, GeV.
if np.nanmax(raw_e) > 10000:
    raw_e_gev = raw_e / 1000.0
else:
    raw_e_gev = raw_e

# FILTRADO FÍSICO (Estricto)
# 1. Z definido y positivo
# 2. Energía > 3 GeV (para tener palanca en la dispersión)
mask = (raw_z > 0) & (np.isfinite(raw_z)) & (raw_e_gev > 3.0)

clean_z = raw_z[mask]
clean_t = raw_t[mask]
clean_e = raw_e_gev[mask]
clean_names = raw_names[mask]

print(f"[DATA] Eventos válidos para análisis de gravedad cuántica: {len(clean_z)}")

if len(clean_z) < 3:
    print("[WARN] Muy pocos eventos (<3). Bajando umbral de energía a 1 GeV...")
    mask = (raw_z > 0) & (np.isfinite(raw_z)) & (raw_e_gev > 1.0)
    clean_z = raw_z[mask]
    clean_t = raw_t[mask]
    clean_e = raw_e_gev[mask]
    clean_names = raw_names[mask]
    print(f"[DATA] Nuevos eventos válidos: {len(clean_z)}")

# --- BLOQUE 4: CÁLCULO K-FACTOR Y TEST LIV ---
print("[PHYS] Calculando factores cosmológicos...")

# Cálculo del K-Factor (Integral cosmológica)
# Aproximación numérica robusta
def get_k_factor(z):
    # K = integral_0^z (1+z') / sqrt(Om*(1+z')^3 + Ol) dz'
    # Usamos un grid precalculado para velocidad
    z_grid = np.linspace(0, z, 100)
    E_inv = 1.0 / np.sqrt(cosmo.Om0 * (1+z_grid)**3 + cosmo.Ode0)
    integrand = (1 + z_grid) * E_inv
    return np.trapz(integrand, z_grid)

# Vectorizamos la función para aplicar al array entero
vec_get_k = np.vectorize(get_k_factor)
K_factors = vec_get_k(clean_z)

# Variable observable: Retraso normalizado (s / GeV)
# Asumimos que el 'trigger' es t=0. Si t_high es el tiempo de llegada,
# t_high es el retraso relativo al frente de onda de baja energía.
lags_norm = clean_t / clean_e

# Regresión Lineal (Buscando la pendiente m_QG)
def model_liv(k, m, c):
    return m * k + c

# Ajuste robusto (Límite de outliers simples)
# Quitamos puntos absurdos (ej: retrasos de 1000s por procesos astrofísicos lentos)
# Nos quedamos con los "prompt" (<100s normalizados) para ser conservadores
mask_outliers = np.abs(lags_norm) < 100
popt, pcov = curve_fit(model_liv, K_factors[mask_outliers], lags_norm[mask_outliers])

m_fit = popt[0]
c_fit = popt[1]
m_err = np.sqrt(np.diag(pcov))[0]

print(f"\n--- RESULTADOS DEL ANÁLISIS LIV ---")
print(f"Pendiente (m): {m_fit:.4e} +/- {m_err:.4e} s/GeV")
# Estimación de la escala de energía QG (si m > 0)
# m ~ 1/H0 * 1/M_QG.  H0 ~ 70 km/s/Mpc ~ 2.3e-18 s^-1
# M_QG ~ 1 / (m * H0)
H0_s = cosmo.H0.to(u.s**-1).value
if m_fit > 0:
    M_QG_est = 1.0 / (m_fit * H0_s) # GeV
    print(f"Escala M_QG Inferida: ~{M_QG_est:.2e} GeV")
else:
    print("Escala M_QG: Indeterminada (Pendiente negativa o cero)")

# --- BLOQUE 5: VISUALIZACIÓN ---
plt.figure(figsize=(12, 7))
plt.style.use('dark_background')

# Datos
plt.errorbar(K_factors[mask_outliers], lags_norm[mask_outliers],
             yerr=0.1*np.abs(lags_norm[mask_outliers])+0.01, fmt='o',
             color='#00ffcc', ecolor='gray', alpha=0.8, label='Datos Fermi-LAT')

# Ajuste
x_plot = np.linspace(0, max(K_factors)*1.1, 50)
plt.plot(x_plot, model_liv(x_plot, m_fit, c_fit),
         color='#ff00ff', linestyle='--', linewidth=2, label=f'Fit LIV (m={m_fit:.2e})')

# Einstein
plt.axhline(y=0, color='white', linestyle=':', label='Einstein (m=0)')

plt.xlabel(r'Factor Cosmológico $K(z)$')
plt.ylabel(r'Retraso Normalizado $\Delta t / E$ (s/GeV)')
plt.title('TIME-OF-FLIGHT TEST: Invariancia de Lorentz en Rayos Gamma\nDatos: Fermi-LAT 3rd GRB Catalog (Multi-Target Updated)', color='white') # Updated plot title
plt.legend()
plt.grid(True, alpha=0.2)

# Etiquetas para los eventos más extremos
if len(K_factors[mask_outliers]) > 0:
    idx_max = np.argmax(K_factors[mask_outliers])
    plt.annotate(f"z={clean_z[mask_outliers][idx_max]:.2f}",
                 (K_factors[mask_outliers][idx_max], lags_norm[mask_outliers][idx_max]),
                 color='yellow', xytext=(-20, 10), textcoords='offset points')

plt.tight_layout()
plt.show()

[SYSTEM] Iniciando Protocolo ASTRO-MINER v3 (Multi-Target)...
[NET] Iniciando barrido de servidores FSSC (2 objetivos)...
   -> Probando: gll_grb_cat.fit ... FALLO [404]
   -> Probando: gll_grb_cat_v05.fit ... FALLO [404]

[CRITICAL ERROR] Ninguna URL del catálogo respondió. La NASA puede estar offline o haber reestructurado todo el directorio.
[ACTION] Verifica manualmente: https://fermi.gsfc.nasa.gov/ssc/data/access/lat/msl_grb_catalog/


SystemExit: 

In [18]:
# --- BLOQUE 0: INFRAESTRUCTURA ---
import sys
import subprocess
import warnings

# Silenciar advertencias de FITS
warnings.filterwarnings('ignore')

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

try:
    import astropy
    import scipy
    import pandas
    import requests
except ImportError:
    print("[SYSTEM] Instalando librerías astronómicas...")
    install("astropy")
    install("scipy")
    install("matplotlib")
    install("pandas")
    install("requests")

import numpy as np
import matplotlib.pyplot as plt
import requests
import io
from astropy.io import fits
from astropy.cosmology import Planck18 as cosmo
from scipy.optimize import curve_fit
from astroquery.heasarc import Heasarc # Asegúrate de que astroquery está importado

print("[SYSTEM] Iniciando Protocolo ASTRO-MINER v4 (HEASARC FTP)...")

# --- BLOQUE 1: ESTRATEGIA DE CONEXIÓN MULTI-SERVIDOR ---
# La NASA aloja los datos en FSSC (Web) y HEASARC (FTP/Data).
# Atacamos ambos frentes. El objetivo es el "LAT 2nd GRB Catalog" (10 años de datos).

# Lista de URLs candidatas (eliminadas ya que ahora usamos astroquery)
# Se mantiene el código como ejemplo de fallback si astroquery falla por completo o para datos específicos.
CANDIDATE_URLS = [
    # Opción A: Servidor HEASARC (Suele ser más estable para scripts)
    "https://heasarc.gsfc.nasa.gov/FTP/fermi/data/lat/catalogs/grb/gll_grb_cat_v02.fit",
    "https://heasarc.gsfc.nasa.gov/FTP/fermi/data/lat/catalogs/2nd_grb/gll_grb_cat_v02.fit",

    # Opción B: Servidor FSSC (Rutas web estándar)
    "https://fermi.gsfc.nasa.gov/ssc/data/access/lat/2nd_GRB_catalog/gll_grb_cat_v02.fit",
    "https://fermi.gsfc.nasa.gov/ssc/data/access/lat/2nd_GRB_catalog/gll_grb_cat_v01.fit",
    "https://fermi.gsfc.nasa.gov/ssc/data/access/lat/2nd_GRB_catalog/gll_grb_cat_v00.fit",

    # Opción C: Ruta genérica antigua (por si acaso)
    "https://fermi.gsfc.nasa.gov/ssc/data/access/lat/grb_catalog/gll_grb_cat_v02.fit"
]

hdul = None
data = None
used_url = ""

print(f"[NET] Intentando acceder al catálogo via Astroquery...")

# Intento con Astroquery (método preferido)
try:
    heasarc = Heasarc()
    heasarc.mission = 'fermi'
    # Usamos query_tap con una consulta SQL para la tabla 'fermilgrb'
    # Esta es la tabla del catálogo GRB de Fermi LAT
    full_table = heasarc.query_tap("SELECT * FROM fermilgrb")
    data = full_table.to_pandas()
    print("[SUCCESS] Datos recuperados con Astroquery desde 'fermilgrb'.")
    # Las columnas de Astroquery suelen estar en minúsculas. Normalizamos a mayúsculas para consistencia.
    data.columns = [c.upper() for c in data.columns]

except Exception as e_astroquery:
    print(f"[ERROR] Fallo en Astroquery: {e_astroquery}. Intentando con URLs directas como fallback...")
    for url in CANDIDATE_URLS:
        try:
            filename = url.split('/')[-1]
            print(f"   -> Target: {url} ...", end=" ")

            response = requests.get(url, timeout=10)

            if response.status_code == 200:
                if b'<!DOCTYPE html>' in response.content[:100]:
                    print("FALLO (Es una web, no un archivo)")
                    continue

                hdul = fits.open(io.BytesIO(response.content))
                data = hdul[1].data
                used_url = url
                print("¡CONEXIÓN ESTABLECIDA!")
                break
            else:
                print(f"STATUS {response.status_code}")

        except Exception as e_request:
            print(f"ERROR ({str(e_request)})")

if data is None or len(data) == 0:
    print("\n[CRITICAL] No se pudo acceder al catálogo desde ninguna fuente.")
    print("Intenta visitar https://heasarc.gsfc.nasa.gov/db-perl/W3Browse/w3table.pl?database=fermi&table=fermilgrb para verificar la disponibilidad.")
    sys.exit()

# --- BLOQUE 2: AUTO-DIAGNÓSTICO DE ESTRUCTURA ---
# Escaneamos las columnas para saber cómo se llaman HOY (la NASA cambia nombres)
cols = [c.upper() for c in data.columns] # Aseguramos que todas las columnas estén en mayúsculas
print(f"[INFO] Estructura detectada: {len(cols)} columnas.")

# Buscamos las columnas críticas heurísticamente
def find_col(keywords, available):
    for k in keywords:
        for col in available:
            if k in col: return col # Match parcial
    return None

col_map = {
    'Z': find_col(['REDSHIFT', 'Z_SPEC', 'Z'], cols),
    'NAME': find_col(['NAME', 'GRB_NAME', 'GRB'], cols),
    # Para LIV, necesitamos el fotón de mayor energía
    'E_MAX': find_col(['HIGHEST_ENERGY', 'MAX_ENERGY', 'ENERGY_PEAK'], cols), # MeV
    'T_MAX': find_col(['HIGHEST_ENERGY_TIME', 'TIME_MAX', 'TPOS'], cols) # s
}

print(f"[INFO] Mapeo: Z->{col_map['Z']}, E->{col_map['E_MAX']}, T->{col_map['T_MAX']}")

if not all(col_map.values()):
    missing_cols = [key for key, value in col_map.items() if value is None]
    print(f"[ERROR] Faltan columnas críticas ({', '.join(missing_cols)}). Estructura del FITS/Tabla irreconocible.")
    print("[DEBUG] Columnas disponibles en el catálogo:", cols)
    sys.exit()

# --- BLOQUE 3: FÍSICA (TIME OF FLIGHT) ---
raw_z = data[col_map['Z']]
raw_e = data[col_map['E_MAX']]
raw_t = data[col_map['T_MAX']]
raw_n = data[col_map['NAME']]

# Limpieza de datos
# 1. Redshift válido
# 2. Energía > 10 GeV (10000 MeV) para ver efectos cuánticos
# 3. Outliers de tiempo (evitar retrasos de horas que son astrofísicos)

# Se asume que HIGHEST_ENERGY está en MeV. Si find_col devuelve ENERGY_PEAK de GBM, las unidades pueden variar.
# Se realiza una conversión a GeV solo si parece ser MeV (valores grandes).
if np.nanmax(raw_e) > 1000:
    raw_e_gev = raw_e / 1000.0
else:
    raw_e_gev = raw_e # Asumimos que ya está en GeV o es un proxy.


valid_mask = (raw_z > 0) & (np.isfinite(raw_z)) & (raw_e_gev > 10) & (raw_t < 100)

clean_z = raw_z[valid_mask]
clean_e_gev = raw_e_gev[valid_mask]
clean_t = raw_t[valid_mask]
clean_names = raw_n[valid_mask]

print(f"[DATA] GRBs útiles (z>0, E>10GeV): {len(clean_z)}")

if len(clean_z) < 3:
    print("[WARN] Pocos eventos extremos. Relajando corte de energía a 3 GeV...")
    valid_mask = (raw_z > 0) & (np.isfinite(raw_z)) & (raw_e_gev > 3) & (raw_t < 100)
    clean_z = raw_z[valid_mask]
    clean_e_gev = raw_e_gev[valid_mask]
    clean_t = raw_t[valid_mask]
    clean_names = raw_n[valid_mask]
    print(f"[DATA] GRBs útiles (Relaxed): {len(clean_z)}")

if len(clean_z) < 2:
    print("[CRITICAL] No hay suficientes datos útiles para el análisis después del filtrado. Ajusta los criterios de filtrado o busca una fuente de datos diferente.")
    sys.exit()

# Cálculo K-Factor (Distancia cosmológica)
print("[PHYS] Calculando integral cosmológica (Factor K)...")
def get_k(z):
    # K = integral_0^z (1+z') / H(z') dz'
    # Usamos aproximación numérica rápida
    z_grid = np.linspace(0, z, 50)
    # H(z) normalizado a H0
    Ez = np.sqrt(cosmo.Om0*(1+z_grid)**3 + cosmo.Ode0)
    integrand = (1+z_grid) / Ez
    return np.trapz(integrand, z_grid)

vec_k = np.vectorize(get_k)
K_factors = vec_k(clean_z)

# Observable: Retraso por unidad de energía (s/GeV)
lags = clean_t / clean_e_gev

# Regresión
def liv_model(k, m_qg_inv, c):
    return m_qg_inv * k + c

popt, pcov = curve_fit(liv_model, K_factors, lags)
m_fit = popt[0]
m_err = np.sqrt(np.diag(pcov))[0]

# --- BLOQUE 4: VISUALIZACIÓN ---
plt.figure(figsize=(12, 7))
plt.style.use('dark_background')

plt.errorbar(K_factors, lags, yerr=0.1*lags, fmt='o', color='#00ffcc',
             ecolor='gray', alpha=0.8, label='GRBs Fermi-LAT')

x_plot = np.linspace(0, max(K_factors)*1.1, 50)
plt.plot(x_plot, liv_model(x_plot, m_fit, popt[1]),
         color='#ff00ff', linestyle='--', label=f'Ajuste LIV (m={m_fit:.2e})')

plt.axhline(0, color='white', linestyle=':', label='Límite Einstein')

# Etiquetar al rey: GRB 090510 (si está)
for i, name in enumerate(clean_names):
    if '090510' in str(name) or clean_e_gev[i] > 20:
        plt.annotate(f"{name}\n{clean_e_gev[i]:.1f}GeV",
                     (K_factors[i], lags[i]), color='yellow', xytext=(5,5), textcoords='offset points')

plt.xlabel('Factor K (Distancia)')
plt.ylabel('Retraso / Energía (s/GeV)')
plt.title('Cazando Violación de Lorentz en GRBs\nDatos: HEASARC API', color='white') # Título generalizado
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n[RESULT] Pendiente LIV detectada: {m_fit:.4e} +/- {m_err:.4e} s/GeV")
if abs(m_fit) < 2*m_err:
    print("CONCLUSIÓN: Compatible con Relatividad General (m=0).")
else:
    print("CONCLUSIÓN: ¡Posible señal de anomalía dimensional!")

[SYSTEM] Iniciando Protocolo ASTRO-MINER v4 (HEASARC FTP)...
[NET] Intentando acceder al catálogo via Astroquery...
[ERROR] Fallo en Astroquery: 'TAPResults' object has no attribute 'to_pandas'. Intentando con URLs directas como fallback...
   -> Target: https://heasarc.gsfc.nasa.gov/FTP/fermi/data/lat/catalogs/grb/gll_grb_cat_v02.fit ... STATUS 404
   -> Target: https://heasarc.gsfc.nasa.gov/FTP/fermi/data/lat/catalogs/2nd_grb/gll_grb_cat_v02.fit ... STATUS 404
   -> Target: https://fermi.gsfc.nasa.gov/ssc/data/access/lat/2nd_GRB_catalog/gll_grb_cat_v02.fit ... STATUS 404
   -> Target: https://fermi.gsfc.nasa.gov/ssc/data/access/lat/2nd_GRB_catalog/gll_grb_cat_v01.fit ... STATUS 404
   -> Target: https://fermi.gsfc.nasa.gov/ssc/data/access/lat/2nd_GRB_catalog/gll_grb_cat_v00.fit ... STATUS 404
   -> Target: https://fermi.gsfc.nasa.gov/ssc/data/access/lat/grb_catalog/gll_grb_cat_v02.fit ... STATUS 404

[CRITICAL] No se pudo acceder al catálogo desde ninguna fuente.
Intenta visitar http

SystemExit: 

In [21]:
# --- BLOQUE 0: INFRAESTRUCTURA PROFESIONAL ---
import sys
import subprocess
import warnings

warnings.filterwarnings('ignore')

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Instalación de la "Llave Maestra" de la Astrofísica: Astroquery
try:
    from astroquery.heasarc import Heasarc
    import astropy
    import scipy
    import matplotlib
    import pandas
except ImportError:
    print("[SYSTEM] Desplegando Astroquery (Acceso API HEASARC)...")
    install("astroquery")
    install("astropy")
    install("scipy")
    install("matplotlib")
    install("pandas")
    from astroquery.heasarc import Heasarc

import numpy as np
import matplotlib.pyplot as plt
from astropy.cosmology import Planck18 as cosmo
from scipy.optimize import curve_fit
from astropy import units as u # Necesario para u.s

print("[SYSTEM] Conectando al mainframe HEASARC (NASA/GSFC)...")

# --- BLOQUE 1: CONSULTA SQL SOBERANA ---
# En lugar de buscar archivos .fit perdidos, consultamos la base de datos.
heasarc = Heasarc()

# Misión: FERMI
# Tabla Objetivo: 'fermilgrb' (Fermi LAT GRB Catalog)
# Esta tabla contiene los fotones GeV críticos para LIV.
TARGET_TABLE = 'fermilgrb'

print(f"[QUERY] Intentando consultar la tabla {TARGET_TABLE}...")

try:
    # La forma correcta de consultar una tabla específica es usar query_tap.
    # Primero, se establece la misión para el objeto Heasarc.
    heasarc.mission = 'fermi'
    # Usamos query_tap con una consulta SQL estándar.
    full_table = heasarc.query_tap(f'SELECT * FROM {TARGET_TABLE}')
    print(f"[SUCCESS] Tabla {TARGET_TABLE} recuperada. Registros totales: {len(full_table)}")

except Exception as e:
    print(f"[ERROR] Fallo al consultar la tabla {TARGET_TABLE}: {e}")
    print("[FALLBACK] Intentando con el catálogo GBM (fermigbrst)....")
    try:
        heasarc.mission = 'fermi'
        TARGET_TABLE = 'fermigbrst' # Actualiza a la tabla GBM
        full_table = heasarc.query_tap(f'SELECT * FROM {TARGET_TABLE}')
        print(f"[SUCCESS] Tabla GBM recuperada. Registros: {len(full_table)}")
    except Exception as e_gbm:
        print(f"[CRITICAL] HEASARC está offline o la tabla {TARGET_TABLE} no es accesible: {e_gbm}")
        sys.exit()

# --- BLOQUE 2: PROCESAMIENTO Y FÍSICA ---
# Convertimos a Pandas o usamos Astropy Table directamente
df = full_table.to_table().to_pandas()

# Normalización de nombres de columnas (HEASARC las devuelve en minúsculas a veces)
df.columns = [c.upper() for c in df.columns]

# Helper function to find column names robustly
def find_col(keywords, available_columns):
    for k in keywords:
        for col in available_columns:
            if k == col or k in col: # Exact match or substring match
                return col
    return None

print(f"[INFO] Columnas disponibles en la tabla: {list(df.columns)}")

# Centralized column mapping using find_col for all critical columns
col_map = {
    'REDSHIFT': find_col(['REDSHIFT', 'Z_SPEC', 'Z'], df.columns),
    'GRB_NAME': find_col(['GRB_NAME', 'NAME'], df.columns),
    # Added EXT_EMISSION_MAX_ENE and EXT_EMISSION_MAX_ENE_T for specific table columns
    'ENERGY': find_col(['HIGHEST_ENERGY', 'ENERGY_MAX', 'PEAK_ENERGY', 'ENERGY', 'EXT_EMISSION_MAX_ENE', 'LIKE_LAT_FLUX_ENE'], df.columns),
    'TIME': find_col(['HIGHEST_ENERGY_TIME', 'TIME_MAX', 'TPOS', 'TIME', 'EXT_EMISSION_MAX_ENE_T', 'LIKE_LAT_T0'], df.columns)
}

# Check for critical column availability
missing_critical_cols = [k for k, v in col_map.items() if v is None]
if missing_critical_cols:
    print(f"[CRITICAL ERROR] Faltan columnas esenciales para el análisis: {', '.join(missing_critical_cols)}")
    print(f"Columnas disponibles en la tabla: {list(df.columns)}")
    sys.exit()

z_col = col_map['REDSHIFT']
name_col = col_map['GRB_NAME']
e_col = col_map['ENERGY']
t_col = col_map['TIME']

# Determine mode based on *found* energy column
if 'EXT_EMISSION_MAX_ENE' in e_col or 'HIGHEST_ENERGY' in e_col:
    print("[MODE] Modo Alta Energía (LAT). Sensibilidad LIV: MÁXIMA.")
else:
    print("[MODE] Modo Baja Energía (GBM-like). Sensibilidad LIV: BAJA. (Puede que falten fotones HE individuales)")
    print("[WARN] Usando columnas generales como proxy de energía/tiempo. Los resultados LIV serán menos fiables.")
    # If it fell back to a generic 'ENERGY' or 'TIME', or other GBM-related, it will use them.
    # The rest of the code will attempt to proceed with these.

# Filtrado Físico
# 1. Redshift > 0
# 2. Datos válidos (no nulos)
df_clean = df[ (df[z_col] > 0) & (df[e_col].notna()) & (df[t_col].notna()) ].copy()

# Conversión de unidades (MeV -> GeV)
# We assume EXT_EMISSION_MAX_ENE is in MeV. If it's LIKE_LAT_FLUX_ENE, it's already in GeV.
if e_col == 'EXT_EMISSION_MAX_ENE':
    df_clean['E_GeV'] = df_clean[e_col] / 1000.0
elif 'FLUX_ENE' in e_col: # e.g. LIKE_LAT_FLUX_ENE is usually in MeV
    df_clean['E_GeV'] = df_clean[e_col] / 1000.0 # Assuming MeV for Flux Energy columns too
    print(f"[WARN] Columna de energía '{e_col}' tratada como MeV para conversión a GeV.")
else:
    # If it's not explicitly MeV, assume it's already in GeV or is a flux/count (less ideal for LIV E dependence)
    df_clean['E_GeV'] = df_clean[e_col]
    print(f"[WARN] Columna de energía '{e_col}' no reconocida como MeV/GeV. Asumiendo que ya está en GeV o es un proxy.")


# Aplicar un filtro de energía mínima para LIV, si la energía es utilizable.
if 'EXT_EMISSION_MAX_ENE' in e_col or 'HIGHEST_ENERGY' in e_col: # Only apply high-energy cut if we have proper LAT energy
    initial_count = len(df_clean)
    df_clean = df_clean[df_clean['E_GeV'] > 3.0] # Filtro > 3 GeV para LAT
    if len(df_clean) < 3:
        print("[WARN] Pocos eventos > 3 GeV. Relajando corte de energía a 1 GeV para LAT...")
        # Ensure we don't try to index with a boolean mask if df_clean is empty after initial filter
        if not df_clean.empty:
            df_clean = df_clean[df_clean['E_GeV'] > 1.0]
    if len(df_clean) < 3:
        print("[WARN] Aún pocos eventos en LAT. Considerar ajustar los umbrales de energía.")
else: # For GBM-like data, a simple energy cut might not be directly comparable.
    # We could filter by peak flux or T90 for general 'energetic' events if needed.
    pass

print(f"[DATA] Eventos con Redshift y Datos Espectrales filtrados: {len(df_clean)}")

if len(df_clean) < 3:
    print("[ERROR] Insuficientes datos para regresión. La base de datos devolvió registros vacíos o muy filtrados.")
    sys.exit()

# --- BLOQUE 3: FACTOR K Y REGRESIÓN LIV ---
print("[PHYS] Integrando cosmología Planck18...")

def get_k_factor(z):
    z_grid = np.linspace(0, z, 50)
    Ez = np.sqrt(cosmo.Om0*(1+z_grid)**3 + cosmo.Ode0)
    integrand = (1+z_grid) / Ez
    return np.trapz(integrand, z_grid)

df_clean['K'] = df_clean[z_col].apply(get_k_factor)

# Observable: Lag normalizado
# Hipótesis: El tiempo de llegada del fotón más energético es el 'retraso' relativo al inicio
df_clean['Lag_Norm'] = df_clean[t_col] / df_clean['E_GeV']

# Limpieza de Outliers (Astrofísica vs Física Fundamental)
# Quitamos eventos donde el retraso es absurdo (>100 s/GeV), probablemente origen no-GRB
mask = np.abs(df_clean['Lag_Norm']) < 100 # Use absolute value for potential negative lags
data_fit = df_clean[mask]

# Modelo Lineal: y = m*x + c
def liv_model(k, m, c):
    return m * k + c

# Asegurarse de que data_fit no esté vacía después de los filtros
if len(data_fit) < 2:
    print("[ERROR] Demasiados outliers eliminados o muy pocos datos para el ajuste. Ajuste los criterios de filtrado.")
    sys.exit()

popt, pcov = curve_fit(liv_model, data_fit['K'], data_fit['Lag_Norm'])
m_fit = popt[0]
m_err = np.sqrt(np.diag(pcov))[0]

# --- BLOQUE 4: VISUALIZACIÓN DE LA VERDAD ---
plt.figure(figsize=(12, 7))
plt.style.use('dark_background')

plt.scatter(data_fit['K'], data_fit['Lag_Norm'], c=data_fit['E_GeV'], cmap='viridis', s=50, alpha=0.9, label='Datos HEASARC')
plt.colorbar(label='Energía (GeV)')

x_plot = np.linspace(0, data_fit['K'].max()*1.1, 50)
plt.plot(x_plot, liv_model(x_plot, m_fit, popt[1]),
         color='#ff00ff', linestyle='--', label=f'Ajuste LIV (m={m_fit:.2e})')

plt.axhline(0, color='white', linestyle=':', label='Límite Einstein')

# Etiquetar al rey: GRB 090510 (si está)
# Filter to avoid KeyError if 'name_col' is not present in data_fit after filtering
if name_col in data_fit.columns:
    top_events = data_fit.nlargest(3, 'E_GeV')
    for _, row in top_events.iterrows():
        plt.annotate(f"{row[name_col]}\n{row['E_GeV']:.1f}GeV",
                     (row['K'], row['Lag_Norm']), color='cyan', xytext=(5,5), textcoords='offset points')

plt.xlabel('Factor K (Distancia)')
plt.ylabel('Retraso / Energía (s/GeV)')
plt.title(f'ANÁLISIS SOBERANO DE INVARIANCIA DE LORENTZ\nFuente: HEASARC API ({TARGET_TABLE})', color='white') # Título generalizado
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n[CONCLUSION] Pendiente m = {m_fit:.4e} +/- {m_err:.4e}")
if m_fit > 2*m_err:
    print(">> INDICIO DE VIOLACIÓN: Pendiente positiva detectada.")
    # Estimación M_QG
    H0_s = cosmo.H0.to(1/u.s).value
    M_QG = 1 / (m_fit * H0_s)
    print(f">> Escala Gravedad Cuántica sugerida: ~{M_QG:.1e} GeV")
elif m_fit < -2*m_err:
    print(">> ANOMALÍA NEGATIVA: Pendiente negativa significativa. Comprobar la física.\n(Esto es inesperado en la mayoría de modelos LIV standard)")
else:
    print(">> COMPATIBLE CON EINSTEIN: No hay evidencia de retraso sistemático.")

[SYSTEM] Conectando al mainframe HEASARC (NASA/GSFC)...
[QUERY] Intentando consultar la tabla fermilgrb...
[SUCCESS] Tabla fermilgrb recuperada. Registros totales: 231
[INFO] Columnas disponibles en la tabla: ['__ROW', 'NAME', 'GCN_NAME', 'TIME', 'TRIGGER_MET', 'RA', 'DEC', 'LII', 'BII', 'ERROR_RADIUS', 'REDSHIFT', 'LUMINOSITY_DISTANCE', 'THETA', 'ZENITH', 'ARR_FLAG', 'DISTANCE_TO_CLOSEST', 'IRFS', 'LLE_BBBD_SIG', 'LLE_BBBD_SIG_DETECTED', 'LLE_T05', 'LLE_T90', 'LLE_T95', 'TL0', 'TL100', 'TL100_ERROR', 'TL1', 'TL0_LOWER', 'TL1_UPPER', 'LIKE_BEST_T0', 'LIKE_BEST_T1', 'LIKE_BEST_TS', 'LIKE_BEST_FLUX', 'LIKE_BEST_FLUX_ERROR', 'LIKE_BEST_FLUX_ENE', 'LIKE_BEST_FLUX_ENE_ERROR', 'LIKE_BEST_FLUENCE', 'LIKE_BEST_FLUENCE_ERROR', 'LIKE_BEST_GRBINDEX', 'LIKE_BEST_GRBINDEX_ERROR', 'LIKE_BEST_EISO_RF', 'LIKE_BEST_EISO_RF_ERROR', 'LIKE_LAT_T0', 'LIKE_LAT_T1', 'LIKE_LAT_TS', 'LIKE_LAT_FLUX', 'LIKE_LAT_FLUX_ERROR', 'LIKE_LAT_FLUX_ENE', 'LIKE_LAT_FLUX_ENE_ERROR', 'LIKE_LAT_FLUENCE', 'LIKE_LAT_FLUENCE_ERR

SystemExit: 